# 02.3 Inlet Profiles: Exercises

## Contents

**Tools:** CADET-GUI widgets: process editor and event timeline; CADET-Process (Python).

**Python used:** Copied event code, parameter changes, slope calculations and plotting.

**Practises:** concentration events, gradient slopes, code export and `plot_events()`.

1. Build the inlet profile
2. Reproduce it in Python
3. Shorten the gradient

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

Translate a laboratory method into events, then reuse and change its Python code.

## Starter model

Salt and Protein flow directly from Inlet to Outlet at **6 mL/min**. The method lasts **300 s**; no column is needed to inspect the inlet profile.

```{figure} ../resources/first_steps/IO.png
:width: 40%
:alt: Inlet connected to Outlet, with flow from left to right.
```

```{tip}
Run the cell below once with **Shift+Enter** to set up the supplied model for this exercise.
```

In [ ]:
# Show Matplotlib plots inside the notebook.
%matplotlib inline
import matplotlib.pyplot as plt

# Load the shared setup functions and create the starter model without events.
%run ../resources/first_steps/lesson_setup.py
from CADETProcess.processModel import Process

flow_sheet, process = inlet_exercise_model()

In [ ]:
# Solution branch only: load the completed events into the process editor.
populate_inlet_exercise_events(process)

## Exercise 1: build the inlet profile

**Load** with **20 mM salt** and **1 mM protein** for **30 s**. **Wash** with **20 mM salt**, without protein, for **60 s**. **Elute** by increasing salt linearly from **20 to 200 mM** over **120 s**, still without protein. Finish with a **final wash** at **200 mM salt**, without protein, until the method ends at **300 s**.

Create the four concentration events on `flow_sheet.inlet.c`. Choose their start times and salt slope from this method. Concentrations in mM have the same numerical values as mol/m³.

```{tip}
**Reminder:** for a linear gradient, slope = (end concentration − start concentration) / gradient duration. With concentrations in mM and duration in seconds, the slope is in mM/s. Set $a_0$ to the start concentration and $a_1$ to this slope; leave $a_2$ and $a_3$ at zero.
```

Check the timeline: does protein stop after loading, and does salt reach 200 mM and then stay constant?

```{tip}
Run the cell below with **Shift+Enter** to open the process editor. Use **New event** and **Add event** to create events; use **Save changes** to edit an existing event. Enter constant values or polynomial coefficients for Salt and Protein.
```

In [ ]:
from cadetgui import ProcessWidget

ProcessWidget(process).display()

```{dropdown} Answer
| Event | Time / s | Salt | Protein |
|---|---:|---|---:|
| load | 0 | constant 20 | 1 |
| wash | 30 | constant 20 | 0 |
| elute | 90 | start 20, slope (200 − 20)/120 = 1.5 mM/s | 0 |
| final_wash | 210 | constant 200 | 0 |

Protein stops when washing starts at 30 s. Salt rises during elution, reaches 200 mM at 210 s and then stays constant.
```

## Exercise 2: use the same method in Python

Click **Show code** in the process editor and copy the code into the answer cell. Name the process variable `python_process` throughout the copied code, keeping `flow_sheet` and the quoted event paths unchanged. Run it, then compare the supplied plot with the editor timeline.

In [ ]:
# Paste the method from Show code in the process editor
python_process = Process(flow_sheet, "inlet_python")
python_process.cycle_time = 300
python_process.add_event("load", "flow_sheet.inlet.c", [20, 1], 0)
python_process.add_event("wash", "flow_sheet.inlet.c", [20, 0], 30)
python_process.add_event("elute", "flow_sheet.inlet.c", [[20, 1.5], [0, 0]], 90)
python_process.add_event("final_wash", "flow_sheet.inlet.c", [200, 0], 210)

In [ ]:
# Plot the copied method directly from its events, with time in seconds.
_ = python_process.plot_events(x_axis_in_minutes=False)
plt.show()

```{dropdown} Interpretation
`plot_events()` and the editor timeline show the same profile: both are drawn from the same events. The copied code is a complete, reusable description of the method.
```

## Exercise 3: shorten the gradient in Python

Copy your method code into the answer cell and use `shorter_process` as the process variable. Shorten the gradient to **60 s**, keeping the same salt endpoints and its start time. Update the salt slope and the start of final wash; keep the method duration at **300 s**.

Predict when salt will reach 200 mM, then run your code and the supplied plot to check.

In [ ]:
# 60 s gradient
shorter_process = Process(flow_sheet, "shorter_gradient")
shorter_process.cycle_time = 300
shorter_process.add_event("load", "flow_sheet.inlet.c", [20, 1], 0)
shorter_process.add_event("wash", "flow_sheet.inlet.c", [20, 0], 30)
shorter_process.add_event("elute", "flow_sheet.inlet.c", [[20, 3], [0, 0]], 90)
shorter_process.add_event("final_wash", "flow_sheet.inlet.c", [200, 0], 150)

In [ ]:
# Plot the changed method directly from its events.
_ = shorter_process.plot_events(x_axis_in_minutes=False)
plt.show()

```{dropdown} Answer
The 60 s gradient needs slope (200 − 20)/60 = 3 mM/s. Starting at 90 s, salt reaches 200 mM at 150 s, so final wash moves to 150 s. Load, wash and the method duration stay unchanged.
```